# Google India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.google.com/jobs/results/?location=India

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:36:36
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Google"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Google/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("GOOGLE INDIA JOB SCRAPER")
print("Source: www.google.com/about/careers/applications/jobs/results")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


google_jobs = []
driver = setup_selenium()

try:
    url = "https://www.google.com/about/careers/applications/jobs/results/?location=India"
    driver.get(url)
    time.sleep(10)

    # Wait for results to render (Google uses heavy JS)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[class*='lLd3Je'], li[class*='sMn82b'], [data-id]"))
        )
    except:
        print("  Waiting longer for Google careers to load...")
        time.sleep(10)

    # Scroll to load more jobs (infinite scroll)
    prev_count = 0
    for scroll in range(30):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "lxml")

        # Google uses specific class names for job cards
        cards = soup.select("li[class*='lLd3Je'], li[class*='sMn82b']")
        if not cards:
            cards = soup.select("[data-id], li.result, [class*='job-result']")

        if len(cards) == prev_count and scroll > 3:
            break  # No new results loaded
        prev_count = len(cards)

        if scroll % 5 == 0:
            print(f"  Scroll {scroll+1}: {len(cards)} jobs found so far")

    # Now parse all visible cards
    soup = BeautifulSoup(driver.page_source, "lxml")

    # Google's class names change with every build — use href patterns instead
    # Primary: any anchor pointing to a Google Careers job result page
    job_anchors = soup.select(
        "a[href*='/about/careers/applications/jobs/results/'], "
        "a[href*='/careers/applications/jobs/results/']"
    )

    # Deduplicate by href and build cards from anchor parents
    seen_hrefs = set()
    cards = []
    for a in job_anchors:
        h = a.get("href", "")
        if h and h not in seen_hrefs:
            seen_hrefs.add(h)
            cards.append((a, h))

    # Fallback to old class-based selectors if anchor approach gets nothing
    if not cards:
        for card in soup.select("li[class*='lLd3Je'], li[class*='sMn82b'], [data-id]"):
            link = card.select_one("a[href]")
            h = link.get("href", "") if link else ""
            if h not in seen_hrefs:
                seen_hrefs.add(h)
                cards.append((link or card, h))

    for anchor, href in cards:
        # Walk up to find the card container
        card = anchor.parent or anchor

        title_el = anchor  # The anchor itself usually contains the title
        title = title_el.get_text(strip=True)
        if not title:
            title_el = card.select_one("h3, h2, [class*='QJPWVe'], [class*='title']")
            title = title_el.get_text(strip=True) if title_el else ""

        loc_el = card.select_one("[class*='r0wTof'], [class*='location'], [class*='city']")
        loc = loc_el.get_text(strip=True) if loc_el else "India"

        # Extract job ID from URL: /jobs/results/12345678901-title-slug
        job_id_match = re.search(r"/jobs/results/([\d]+)", href)
        job_id = job_id_match.group(1) if job_id_match else href.split("/")[-1]

        # Build canonical job URL — use the full href so the link goes to the right page
        job_url = href if href.startswith("http") else f"https://www.google.com{href}" if href else ""

        if title and title not in [j["title"] for j in google_jobs]:
            google_jobs.append({
                "job_id": str(job_id),
                "title": title,
                "company_name": "Google",
                "raw_jd_text": card.get_text(" ", strip=True),
                "location_city": loc.split(",")[0].strip(),
                "industry": "Technology",
                "date_posted": datetime.now().strftime("%Y-%m-%d"),
                "is_active": True,
                "job_url": job_url,
                "business_unit": "",
                "source_platform": "Google Careers",
            })

    print(f"  Found {len(google_jobs)} total Google India jobs")

    # Fetch JD for first N jobs — use each job's own stored URL (no scope bug)
    if google_jobs:
        print(f"  Fetching JD details for up to 30 jobs...")
        for i, job in enumerate(google_jobs[:30]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 200:
                continue
            jd_url = job["job_url"]  # use the URL we already stored — correct per job
            if not jd_url:
                jd_url = f"https://www.google.com/about/careers/applications/jobs/results/{job['job_id']}"
            jd = fetch_jd_selenium(driver, jd_url)
            if jd:
                google_jobs[i]["raw_jd_text"] = jd
            if (i + 1) % 10 == 0:
                print(f"    Fetched {i+1}/{min(30, len(google_jobs))} JDs")

except Exception as e:
    print(f"  Error: {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"Total Google India jobs: {len(google_jobs)}")


GOOGLE INDIA JOB SCRAPER
Source: www.google.com/about/careers/applications/jobs/results


  Scroll 1: 20 jobs found so far


  Found 0 total Google India jobs
Total Google India jobs: 0


In [5]:
df_google = save_results(google_jobs, "Google", OUTPUT_DIR)
if df_google is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_google.columns]
    print(df_google[cols].head(10).to_string())


  [WARN] No jobs found for Google
